# 03 · Features and pipeline: each addition judged on June
**Owner: P2** · Status: `[x]`

Rules: stateless features live in `src/features.py`; anything learned lives inside the `Pipeline` (`src/models.py`) and is fitted on **May only**. June scores every choice. **July is not loaded here.** Report the features that lost too.

In [1]:
import sys; sys.path.insert(0, "..")          # repo root, so `import src...` works
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import TARGET
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 140)

import itertools, time
from src.clean import build_datasets
from src.features import FEATURE_SETS
from src.models import make_pipeline, make_model
from src.evaluate import metrics, bootstrap_vs_baseline, save_results

D = build_datasets()
train, val = D["train"], D["val"]            # NB03 never touches D["test"]
del D
ytr, yva = train[TARGET], val[TARGET]
days_va = val.tpep_pickup_datetime.dt.date   # bootstrap blocks
print(len(train), len(val))

def june(model_name, fs, **params):
    pipe = make_pipeline(make_model(model_name, **params), fs).fit(train, ytr)
    return pipe, pipe.predict(val)

133342 133459


## 1. Ablation: feature sets × models (train May, score June)
New sets beyond the skeleton (f0–f4):
- **f5_zones**: pickup and drop-off zone IDs, **target-encoded inside the pipeline** (sklearn `TargetEncoder`, 5-fold cross-fitted on May; June gets May's encodings).
- **f6_pair**: plus the zone *pair*, target-encoded the same way (the zone-pair mean is never computed on June/July).
- **f7_time_x**: plus hour × weekday/weekend, and distance × (time band × weekday/weekend) so the per-mile price can vary with traffic.

Three model families: OLS, a single decision tree (defaults, tuned in §2), and histogram gradient boosting (sklearn, defaults, tuned in §3).

In [2]:
rows = []
for fs in FEATURE_SETS:
    for name in ["linear", "tree", "hgb"]:
        pipe, p = june(name, fs)
        rows.append(dict(feature_set=fs, model=name, n_features=pipe[:-1].transform(val.head(5)).shape[1],
                         **{k: round(v, 3) for k, v in metrics(yva, p).items() if k != "n"}))
ablation = pd.DataFrame(rows)
ablation.to_csv("../results/june_ablation.csv", index=False)
ablation.pivot(index="feature_set", columns="model", values=["mae", "rmse"])

mae                 rmse              
model          hgb linear   tree    hgb linear   tree
feature_set                                          
f0_distance  2.800  2.979  2.793  3.995  4.151  3.981
f1_time      2.499  2.819  2.770  3.639  3.949  3.949
f2_geo       2.444  2.733  2.747  3.562  3.850  3.920
f3_airport   2.441  2.719  2.747  3.557  3.838  3.920
f4_shape     2.434  2.590  2.747  3.546  3.672  3.920
f5_zones     2.259  2.577  2.699  3.344  3.663  3.866
f6_pair      2.190  2.525  2.633  3.243  3.604  3.791
f7_time_x    2.169  2.455  2.536  3.222  3.540  3.667

**Read it (June MAE, \$).**
- **f0 → f1 (hour, weekday):** linear 2.979 → 2.819, HGB 2.800 → 2.499. Time of day matters, through the time charge. The default tree hardly moves (2.793 → 2.770).
- **f1 → f2 (boroughs):** small gains (linear −0.09, HGB −0.06). Borough is weak because 93% of pickups are in Manhattan.
- **f2 → f3 (airport flags):** ≈ 0 for every model (−0.014 / 0.000 / −0.003). It mostly duplicates borough/zone information.
- **f3 → f4 (log distance, distance band):** −0.13 for linear (the fare is concave in distance); nothing for the trees, which already split on distance.
- **f4 → f5 (zone target encoding):** the biggest single gain for HGB (2.434 → 2.259); small for linear (−0.013).
- **f5 → f6 (zone pair):** −0.05 to −0.07 for all three.
- **f6 → f7 (time × distance):** helps the linear model most (−0.07), the tree (−0.10) and HGB a little (−0.02).

An addition that helps the tree but not the linear model (zones) shows interactions the linear model cannot express without explicit interaction terms.

## 2. Tune the tree on June
The skeleton grid's best (depth 12, leaf 50) sat on the edge of the grid, so the grid is extended. Feature set: f7_time_x (best for the tree in §1).

In [3]:
res = []
for depth, leaf in itertools.product([8, 10, 12, 14, 16, 20], [10, 20, 50, 100, 200]):
    _, p = june("tree", "f7_time_x", max_depth=depth, min_samples_leaf=leaf)
    res.append(dict(depth=depth, leaf=leaf, **{k: round(v, 3) for k, v in metrics(yva, p).items() if k != "n"}))
tree_grid = pd.DataFrame(res).sort_values("mae")
tree_grid.to_csv("../results/june_tree_grid.csv", index=False)
tree_grid.head(8)

,depth,leaf,mae,rmse,bias
27,20,50,2.372,3.482,-0.102
22,16,50,2.377,3.486,-0.104
17,14,50,2.389,3.498,-0.106
21,16,20,2.394,3.521,-0.095
26,20,20,2.394,3.522,-0.092
16,14,20,2.399,3.523,-0.095
28,20,100,2.407,3.520,-0.108
12,12,50,2.408,3.516,-0.109


**Read it.** The best setting is depth 20, leaf 50 (MAE 2.372), and depths 14–20 with leaf 50 are all within \$0.02 (2.372–2.389). The optimum is no longer an artefact of the grid edge: the curve is flat there. The tree is carried into §4 at depth 20 / leaf 50.

## 3. Tune gradient boosting on June
Stage 1: loss × learning rate × leaves × iterations × min leaf size (48 fits). `absolute_error` targets the conditional median, which is what MAE rewards; `squared_error` targets the mean. Stage 2 extends past any edge optimum from stage 1.

**Rule, fixed before the grid ran:** because many neighbouring configurations differ by cents, we take the *cheapest* configuration (fewest leaves, then fewest iterations) within \$0.01 of the best June MAE, rather than the noisy argmin.

In [4]:
t0 = time.time(); res = []
for loss, lr, leaves, iters, msl in itertools.product(["squared_error", "absolute_error"], [0.05, 0.1], [31, 63, 127], [300, 600], [20, 100]):
    _, p = june("hgb", "f7_time_x", loss=loss, learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl)
    res.append(dict(loss=loss, learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl,
                    **{k: round(v, 4) for k, v in metrics(yva, p).items() if k != "n"}))
hgb1 = pd.DataFrame(res).sort_values("mae")
print(f"stage 1: {time.time() - t0:.0f}s"); display(hgb1.head(8)); display(hgb1.groupby("loss")["mae"].agg(["min", "median"]))

stage 1: 492s


,loss,learning_rate,max_leaf_nodes,max_iter,min_samples_leaf,mae,rmse,bias
35,absolute_error,0.05,127,600,100,2.0765,3.2269,0.4146
47,absolute_error,0.10,127,600,100,2.0815,3.2275,0.3980
45,absolute_error,0.10,127,300,100,2.0816,3.2341,0.4207
33,absolute_error,0.05,127,300,100,2.0852,3.2451,0.4366
34,absolute_error,0.05,127,600,20,2.0863,3.2433,0.4357
43,absolute_error,0.10,63,600,100,2.0869,3.2415,0.4236
46,absolute_error,0.10,127,600,20,2.0888,3.2381,0.4085
31,absolute_error,0.05,63,600,100,2.0903,3.2518,0.4370


,min,median
loss,,
absolute_error,2.0765,2.09595
squared_error,2.1451,2.15830


In [5]:
t0 = time.time(); res = []
for lr, leaves, iters, msl in itertools.product([0.05, 0.1], [127, 255], [600, 1000], [100, 300]):
    _, p = june("hgb", "f7_time_x", loss="absolute_error", learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl)
    res.append(dict(loss="absolute_error", learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl,
                    **{k: round(v, 4) for k, v in metrics(yva, p).items() if k != "n"}))
hgb = pd.concat([hgb1, pd.DataFrame(res)]).drop_duplicates(subset=["loss", "learning_rate", "max_leaf_nodes", "max_iter", "min_samples_leaf"]).sort_values("mae")
hgb.to_csv("../results/june_hgb_grid.csv", index=False)
print(f"stage 2: {time.time() - t0:.0f}s, best MAE {hgb.mae.min():.4f}, range of the top 10: {hgb.mae.head(10).min():.4f}-{hgb.mae.head(10).max():.4f}")
near = hgb[hgb.mae <= hgb.mae.min() + 0.01].sort_values(["max_leaf_nodes", "max_iter", "mae"])
display(near)
chosen = near.iloc[0][["loss", "learning_rate", "max_leaf_nodes", "max_iter", "min_samples_leaf"]].to_dict()
chosen = {k: (v.item() if hasattr(v, "item") else v) for k, v in chosen.items()}
print("chosen:", chosen)

stage 2: 447s, best MAE 2.0722, range of the top 10: 2.0722-2.0801


,loss,learning_rate,max_leaf_nodes,max_iter,min_samples_leaf,mae,rmse,bias
45,absolute_error,0.10,127,300,100,2.0816,3.2341,0.4207
35,absolute_error,0.05,127,600,100,2.0765,3.2269,0.4146
9,absolute_error,0.10,127,600,300,2.0801,3.2266,0.4045
1,absolute_error,0.05,127,600,300,2.0806,3.2352,0.4241
47,absolute_error,0.10,127,600,100,2.0815,3.2275,0.3980
2,absolute_error,0.05,127,1000,100,2.0753,3.2215,0.4008
3,absolute_error,0.05,127,1000,300,2.0786,3.2284,0.4151
11,absolute_error,0.10,127,1000,300,2.0812,3.2245,0.3966
4,absolute_error,0.05,255,600,100,2.0722,3.2165,0.3920
5,absolute_error,0.05,255,600,300,2.0744,3.2243,0.4181


chosen: {'loss': 'absolute_error', 'learning_rate': 0.1, 'max_leaf_nodes': 127, 'max_iter': 300, 'min_samples_leaf': 100}


**Read it.** `absolute_error` beats `squared_error` everywhere (best 2.077 vs 2.145): it targets the conditional median, which is what MAE rewards. Stage 2 went past the stage-1 edge (127 leaves / 600 iterations / leaf 100). The best is 2.0722 (255 leaves), but the top ten settings span only \$0.008, so the surface is flat. The rule picks the cheapest setting within \$0.01 of the best: **learning rate 0.10, 127 leaves, 300 iterations, min leaf 100 (MAE 2.0816)**. The price of parsimony is \$0.009 of June MAE.

*Process note:* a first run of this notebook stopped at the freeze cell because `config.py` held a hand-picked setting (lr 0.05, 600 iterations) instead of the one the written rule selects. The config was corrected to the rule's choice before NB04 ran.

## 4. Model selection on June, with uncertainty
**Rule, fixed before this section ran:** take the lowest June MAE; if its gain over the best *simpler* candidate has a 95% day-block bootstrap CI that includes 0, take the simpler one. Candidates (each at its best feature set and tuned params): OLS, tuned tree, tuned HGB, plus the rate rule baselines.

The bootstrap resamples whole June days (30 blocks), because trips on the same day share weather, traffic and events. **Caveat:** June has now been used for feature choice, tuning *and* this comparison, so the winner's June MAE is optimistic. July (NB04) is the honest estimate.

In [6]:
from src.models import RateRule, TLC_START, TLC_PER_MILE
best_tree = tree_grid.iloc[0]
cands = {
    "rate rule (TLC $3 + $3.50/mi)": RateRule(TLC_START, TLC_PER_MILE).fit(train, ytr).predict(val),
    "rate rule (fit on May)": RateRule().fit(train, ytr).predict(val),
    "linear f7": june("linear", "f7_time_x")[1],
    f"tree f7 (depth {best_tree.depth}, leaf {best_tree.leaf})": june("tree", "f7_time_x", max_depth=int(best_tree.depth), min_samples_leaf=int(best_tree.leaf))[1],
    "hgb f7 (chosen params)": june("hgb", "f7_time_x", **chosen)[1],
}
sel = pd.DataFrame({k: metrics(yva, p) for k, p in cands.items()}).T.drop(columns="n").round(3)
display(sel)
names = list(cands)
best, simpler = names[-1], names[-2]
cmp = {f"{best} vs {o}": bootstrap_vs_baseline(yva, cands[best], cands[o], groups=days_va, n_boot=2000).loc[["mae_diff", "rmse_diff"]]
       for o in [simpler, "linear f7", "rate rule (fit on May)"]}
pd.concat(cmp).round(3)

,mae,rmse,bias
rate rule (TLC $3 + $3.50/mi),4.894,6.552,4.885
rate rule (fit on May),2.979,4.151,-0.061
linear f7,2.455,3.540,-0.081
"tree f7 (depth 20.0, leaf 50.0)",2.372,3.482,-0.102
hgb f7 (chosen params),2.082,3.234,0.421


point     lo     hi
hgb f7 (chosen params) vs tree f7 (depth 20.0, ... mae_diff  -0.290 -0.312 -0.269
                                                   rmse_diff -0.248 -0.277 -0.219
hgb f7 (chosen params) vs linear f7                mae_diff  -0.374 -0.399 -0.346
                                                   rmse_diff -0.306 -0.342 -0.267
hgb f7 (chosen params) vs rate rule (fit on May)   mae_diff  -0.897 -0.946 -0.842
                                                   rmse_diff -0.916 -1.011 -0.818

**Decision: gradient boosting on f7_time_x.** Its June MAE (2.082) beats the best simpler candidate, the depth-20 tree (2.372), by \$0.29 per trip. The 95% day-block interval of the difference is −\$0.312 to −\$0.269, which excludes 0, so the rule keeps the boosted model. It also beats the linear model (−\$0.37) and the fitted rate rule (−\$0.90, CI −\$0.95 to −\$0.84). The official TLC rule (\$3 + \$3.50/mi) is far worse (MAE 4.89, bias +\$4.89): distance alone misses the time charge.

Note the bias: `absolute_error` predicts the median, so mean(actual − predicted) is **+\$0.42** on June (fares are right-skewed). The quote buffer in NB04 absorbs this.

## 5. Features that lost / did nothing
From the ablation table (§1), June MAE change when the set is added, per model:
| Feature group | linear | tree (default) | HGB (default) | Verdict |
|---|---|---|---|---|
| airport flags (f3) | −0.014 | 0.000 | −0.003 | did nothing; kept only because later sets are cumulative |
| log distance + band (f4) | −0.129 | 0.000 | −0.007 | linear only |
| boroughs (f2) | −0.086 | −0.023 | −0.055 | small |
| zone target encoding (f5) | −0.013 | −0.048 | −0.175 | big for HGB |
| zone pair (f6) | −0.052 | −0.066 | −0.069 | kept |
| hour × weekend, distance × time band (f7) | −0.070 | −0.097 | −0.021 | kept |

No group made a model worse on June.

## 6. Freeze (Gate 2)
**Frozen at Gate 2 (`src/config.py`):** `FINAL_MODEL = "hgb"`, `FINAL_FEATURE_SET = "f7_time_x"`, `FINAL_PARAMS = {loss: absolute_error, learning_rate: 0.1, max_leaf_nodes: 127, max_iter: 300, min_samples_leaf: 100}`.

Justification: lowest June MAE of all candidates; its gain over the best simpler model (the tuned tree) is \$0.29 with a 95% day-block CI that excludes 0; hyperparameters come from a flat region of the grid by the parsimony rule written before the grid ran. The cell below checks that `config.py` matches this selection and writes `results/gate2_freeze.json`.

In [7]:
from src.config import FINAL_FEATURE_SET, FINAL_MODEL, FINAL_PARAMS
assert (FINAL_MODEL, FINAL_FEATURE_SET) == ("hgb", "f7_time_x") and FINAL_PARAMS == chosen, "config.py does not match the June selection"
save_results({"feature_set": FINAL_FEATURE_SET, "model": FINAL_MODEL, "params": FINAL_PARAMS,
              "june_selection_table": sel.to_dict(orient="index"),
              "june_bootstrap": {k: v.to_dict(orient="index") for k, v in cmp.items()}}, "gate2_freeze.json")
print("frozen:", FINAL_FEATURE_SET, FINAL_MODEL, FINAL_PARAMS)

frozen: f7_time_x hgb {'loss': 'absolute_error', 'learning_rate': 0.1, 'max_leaf_nodes': 127, 'max_iter': 300, 'min_samples_leaf': 100}
